# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](assets/warehouse.png)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
%matplotlib inline
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # A partir de la imagen (grid 5x6, fila 0 arriba, columna 0 izquierda):
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,   # entrega
            (2, 2): 2.0,    # carga
            (3, 5): -10.0,  # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        # Piso normal
        self.p_intended = 0.90
        self.p_perpendicular = 0.05

        # Piso resbaloso
        self.p_intended_slippery = 0.60
        self.p_perpendicular_slippery = 0.20

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): recompensa de estar en el estado actual.
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Un estado terminal es absorbente.
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended = self.p_intended_slippery
            p_perp = self.p_perpendicular_slippery
        else:
            p_intended = self.p_intended
            p_perp = self.p_perpendicular

        # Perpendiculares a (dr, dc)
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_perp),
            (perp2, p_perp),
        ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (
                state[0] + next_action[0],
                state[1] + next_action[1],
            )
            # Si sale del grid o golpea una estantería, se queda quieto.
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))

        return transitions



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # Σ_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for _ in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(
                        grid, state, policy[state], V
                    )
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V


def policy_improvement(grid, V):
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria
    policy = {
        state: grid.actions[0]
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []
    for _ in range(max_iter):
        # 2. evaluación
        V = policy_evaluation(grid, policy, threshold=threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)

        changed = new_policy != policy
        history.append(changed)
        policy = new_policy

        # 4. repetir hasta estabilidad
        if not changed:
            break

    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [True, True, True, False]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


## Respuestas — Parte 5 (revisadas)

1. Con los parámetros base, la trayectoria de movimientos deseados llega a carga +2. Al ser estocástico, no hay un destino garantizado: la probabilidad de terminar en carga es aproximadamente 76,33 % y en entrega 23,67 %.
2. Una recompensa menor cercana puede maximizar el retorno al evitar costos de pasos y descuento acumulado.
3. Para aislar el efecto del piso, comparé contra un mundo idéntico con todos los pisos normales. Cambian las acciones en **(1,2) y (3,1)**. No basta enumerar las tres celdas resbalosas: un cambio de dinámica también puede cambiar decisiones en otros estados.
4. El costo −1 penaliza demoras y favorece terminar pronto. La política maximiza el retorno esperado completo, no únicamente la recompensa terminal.
5. La distribución depende del estado actual: 0,90/0,05/0,05 en piso normal y 0,60/0,20/0,20 en resbaloso.

### Predicciones antes de los experimentos

- **A:** con costo −0,1, espero que compense recorrer más pasos hacia +10.
- **B:** con éxito 0,4 y desviaciones 0,3/0,3, espero menor control. Una política puede mantenerse aunque cambien sus valores y probabilidades de destino.
- **C:** con gamma 0,99, espero que aumente el atractivo de +10. Se verifica la ruta completa; observar solo la acción inicial no basta.

Las siguientes celdas ejecutan cada modificación por separado, sin acumular cambios.

In [7]:
def absorption(g,pi):
 states=[s for s in g.states() if not g.is_terminal(s)]; terminals=list(g.terminal_states);idx={s:i for i,s in enumerate(states)}
 Q=np.zeros((len(states),len(states)));R=np.zeros((len(states),len(terminals)))
 for s in states:
  for nxt,p in g.get_transition_probs(s,pi[s]):
   if nxt in idx:Q[idx[s],idx[nxt]]+=p
   else:R[idx[s],terminals.index(nxt)]+=p
 return np.linalg.solve(np.eye(len(states))-Q,R)[idx[g.start]]

base = WarehouseMDP()
v_base, _ = value_iteration(base, threshold=1e-10)
p_base = extract_policy(base, v_base)

def intended_path(g, policy):
    path = [g.start]
    while not g.is_terminal(path[-1]):
        s = path[-1]; a = policy[s]
        nxt = (s[0]+a[0], s[1]+a[1])
        if not g.is_valid_state(nxt): nxt = s
        if nxt in path: return path + [nxt]
        path.append(nxt)
    return path

experiments = {}
for name in ["Base", "Todo normal", "A", "B", "C"]:
    g = WarehouseMDP()
    if name == "Todo normal":
        g.p_intended_slippery = .9; g.p_perpendicular_slippery = .05
    elif name == "A": g.living_reward = -.1
    elif name == "B":
        g.p_intended_slippery = .4; g.p_perpendicular_slippery = .3
    elif name == "C": g.gamma = .99
    v, _ = value_iteration(g, threshold=1e-10)
    policy = extract_policy(g, v)
    experiments[name] = (g, v, policy)
    print(name, "V(START) =", round(v[g.start], 6))
    print("Ruta de movimientos deseados:", intended_path(g, policy))
    print("Estados con acción distinta:", [s for s in p_base if p_base[s] != policy[s]])
    probs = absorption(g, policy)
    assert np.isclose(probs.sum(), 1)
    print("P(entrega, carga, peligro mortal):", np.round(probs, 6))

Base V(START) = -2.574501
Ruta de movimientos deseados: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
Estados con acción distinta: []
P(entrega, carga, peligro mortal): [0.236699 0.763301 0.      ]
Todo normal V(START) = -2.406598
Ruta de movimientos deseados: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]
Estados con acción distinta: [(1, 2), (3, 1)]
P(entrega, carga, peligro mortal): [0.894903 0.105097 0.      ]
A V(START) = 1.648711
Ruta de movimientos deseados: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]
Estados con acción distinta: [(1, 0), (1, 2), (3, 2)]
P(entrega, carga, peligro mortal): [0.747813 0.252187 0.      ]
B V(START) = -2.7062
Ruta de movimientos deseados: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
Estados con acción distinta: []
P(entrega, carga, peligro mortal): [0.405508 0.594492 0.      ]
C V(START) = -1.456021
Ruta de movimientos deseados: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]
Estados con acción distinta:

### Resultados e interpretación

**A:** cambia la decisión en (1,2) de bajar a ir a la derecha; la ruta deseada llega a +10. También cambian (1,0) y (3,2). La probabilidad de entrega sube aproximadamente a 74,78 %.

**B:** ninguna acción óptima cambia, pero V(START) disminuye de −2,575 aproximadamente a −2,706. La probabilidad de entrega sube a 40,55 % por las desviaciones: más llegadas a +10 no implican mayor retorno descontado si se incurre en más demoras y costos.

**C:** **sí cambia la ruta**, por el cambio de acción en (1,2). La acción inicial sigue siendo derecha, pero la ruta deseada ahora llega a entrega. La probabilidad de entrega es aproximadamente 70,90 %. La respuesta anterior confundía que la primera acción fuera igual con que toda la política fuera igual.

### Bonus reproducible

Buscamos el umbral donde cambia la decisión en (1,2), la bifurcación de la ruta desde START. La primera acción en START sigue siendo derecha a ambos lados del umbral. Se usa tolerancia de Bellman más estricta para no confundir error numérico con cambio de política.

In [8]:
def prefers_delivery(living_reward):
    g = WarehouseMDP(); g.living_reward = living_reward
    v, _ = value_iteration(g, threshold=1e-12)
    p = extract_policy(g, v)
    return intended_path(g, p)[-1] == (0, 5)

lo, hi = -1.0, -0.1
assert not prefers_delivery(lo) and prefers_delivery(hi)
for _ in range(35):
    mid = (lo + hi) / 2
    if prefers_delivery(mid): hi = mid
    else: lo = mid
print(f"Cambio de ruta entre {lo:.9f} y {hi:.9f}")

# Comprobaciones del MDP y soluciones, incluyendo fronteras y terminales.
assert len(base.states()) == 26
for state in base.states():
    for action in base.actions:
        transitions = base.get_transition_probs(state, action)
        assert all(base.is_valid_state(t) and p >= 0 for t, p in transitions)
        assert np.isclose(sum(p for _, p in transitions), 1)
assert base.get_transition_probs((0, 5), (0, 1)) == [((0, 5), 1.0)]
assert np.isclose(sum(p for t, p in base.get_transition_probs((0, 0), (-1, 0)) if t == (0, 0)), .95)
pi_check, v_check, _ = policy_iteration(base, threshold=1e-10)
assert max(abs(v_check[s]-v_base[s]) for s in base.states()) < 1e-8
residual = max(abs(v_base[s] - (base.get_reward(s) if base.is_terminal(s) else
    base.get_reward(s) + base.gamma * max(expected_next_value(base,s,a,v_base) for a in base.actions)))
    for s in base.states())
assert residual < 1e-8
print("Pruebas correctas. Residuo de Bellman:", residual)

Cambio de ruta entre -0.798340631 y -0.798340631


Pruebas correctas. Residuo de Bellman: 4.601119485414529e-11


**Convención terminal:** aunque la función de transición representa los terminales como absorbentes, los algoritmos detienen allí el retorno y fijan V(terminal)=R(terminal). La recompensa terminal se cobra una vez; no se aplica indefinidamente la recurrencia de un estado no terminal.